# Pipeline Architecture — LangChain

**Pattern:** ETL — Extract (Python) → Transform (Python) → Load (LLM)

```
Raw strings → extract() → list[dict] → transform() → scored+ranked → load_chain → report
```

Pipeline = mix of pure Python functions and LLM chains. LangChain's `|` operator handles both.

In [ ]:
import os, json
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
RAW = ["Tokyo|Clear|18|Low|10|22:30 JST", "Paris|Partly Cloudy|16|Low|8|15:30 CET", "Bangalore|Rainy|25|Medium|6|20:00 IST"]
print("✓ Ready")

In [ ]:
# EXTRACT: pure Python
def extract(raw):
    records = []
    for line in raw:
        city, weather, temp, safety_level, safety_score, time = line.split("|")
        records.append({"city": city, "weather": weather, "temp_c": int(temp),
                        "safety_level": safety_level, "safety_score": int(safety_score), "local_time": time})
    print(f"  [Extract] {len(records)} records")
    return records

# TRANSFORM: pure Python
def transform(records):
    score_map = {"Clear": 9, "Partly Cloudy": 7, "Rainy": 6, "Cloudy": 5}
    for r in records:
        r["weather_score"] = score_map.get(r["weather"], 5)
        r["combined_score"] = round((r["weather_score"] + r["safety_score"]) / 2, 1)
    records.sort(key=lambda x: x["combined_score"], reverse=True)
    for i, r in enumerate(records, 1): r["rank"] = i
    print(f"  [Transform] ranked")
    return records

# LOAD: LLM chain
load_chain = (ChatPromptTemplate.from_messages([
    ("system","Format structured data into a markdown travel ranking report."),
    ("human","Data: {data}")
]) | llm | (lambda r: r.content))

print("ETL steps defined")

In [ ]:
extracted   = extract(RAW)
transformed = transform(extracted)
report      = load_chain.invoke({"data": json.dumps(transformed, indent=2)})
print("--- Report ---")
print(report)

## Key Takeaways

| Step | Implementation | Why |
|---|---|---|
| Extract | Pure Python | No LLM needed for parsing |
| Transform | Pure Python | Deterministic scoring |
| Load | LLM chain | Narrative formatting benefits from LLM |

**Pipeline = use LLM only where it adds value.** Pure functions are faster, cheaper, and deterministic.

### Next: [LangGraph Pipeline →](../LangGraph/pipeline.ipynb)